In [0]:
print("Spark version:", spark.version)

test_df = spark.range(10)

display(test_df)

In [0]:
from pyspark.sql import functions as F

NUM_CUSTOMERS = 10000

customers = (
    spark.range(1, NUM_CUSTOMERS + 1)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "customer_name",
        F.concat(F.lit("Customer_"), F.col("customer_id"))
    )
    .withColumn(
        "email",
        F.concat(
            F.lit("customer"),
            F.col("customer_id"),
            F.lit("@example.com")
        )
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Mumbai"),
                F.lit("Pune"),
                F.lit("Bangalore"),
                F.lit("Delhi"),
                F.lit("Hyderabad"),
                F.lit("Chennai")
            ),
            (F.rand(seed=42) * 6 + 1).cast("int")
        )
    )
    .withColumn(
        "state",
        F.element_at(
            F.array(
                F.lit("Maharashtra"),
                F.lit("Karnataka"),
                F.lit("Delhi"),
                F.lit("Telangana"),
                F.lit("Tamil Nadu")
            ),
            (F.rand(seed=43) * 5 + 1).cast("int")
        )
    )
    .withColumn("country", F.lit("India"))
    .withColumn(
        "signup_date",
        F.date_sub(
            F.current_date(),
            (F.rand(seed=44) * 1500).cast("int")
        )
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(customers.limit(20))

In [0]:
customers.printSchema()

print("Customer count:", customers.count())

display(customers.limit(10))

In [0]:
from pyspark.sql import functions as F

NUM_PRODUCTS = 500

products = (
    spark.range(1, NUM_PRODUCTS + 1)
    
    # Product ID
    .withColumnRenamed("id", "product_id")
    
    # Product name
    .withColumn(
        "product_name",
        F.concat(
            F.lit("Product_"),
            F.col("product_id")
        )
    )
    
    # Category
    .withColumn(
        "category",
        F.element_at(
            F.array(
                F.lit("Electronics"),
                F.lit("Clothing"),
                F.lit("Home"),
                F.lit("Beauty"),
                F.lit("Sports")
            ),
            (F.rand(seed=50) * 5 + 1).cast("int")
        )
    )
    
    # Subcategory
    .withColumn(
        "subcategory",
        F.element_at(
            F.array(
                F.lit("Premium"),
                F.lit("Standard"),
                F.lit("Basic"),
                F.lit("Accessories")
            ),
            (F.rand(seed=51) * 4 + 1).cast("int")
        )
    )
    
    # Product price
    .withColumn(
        "price",
        F.round(
            F.rand(seed=52) * 4950 + 50,
            2
        )
    )
    
    # Supplier ID
    .withColumn(
        "supplier_id",
        (F.rand(seed=53) * 50 + 1).cast("int")
    )
    
    # Last updated timestamp
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(products.limit(20))

In [0]:
products.printSchema()

print("Product count:", products.count())

display(products.limit(10))

In [0]:
NUM_ORDERS = 5000

orders = (
    spark.range(1, NUM_ORDERS + 1)
    
    # Order ID
    .withColumnRenamed("id", "order_id")
    
    # Assign each order to an existing customer
    .withColumn(
        "customer_id",
        (F.rand(seed=60) * NUM_CUSTOMERS + 1).cast("long")
    )
    
    # Order date within approximately the last 2 years
    .withColumn(
        "order_date",
        F.date_sub(
            F.current_date(),
            (F.rand(seed=61) * 730).cast("int")
        )
    )
    
    # Order status
    .withColumn(
        "order_status",
        F.element_at(
            F.array(
                F.lit("PLACED"),
                F.lit("SHIPPED"),
                F.lit("DELIVERED"),
                F.lit("CANCELLED")
            ),
            (F.rand(seed=62) * 4 + 1).cast("int")
        )
    )
    
    # Order amount
    .withColumn(
        "total_amount",
        F.round(
            F.rand(seed=63) * 9950 + 50,
            2
        )
    )
    
    # Last updated timestamp
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(orders.limit(20))

In [0]:
NUM_ORDER_ITEMS = 15000

order_items = (
    spark.range(1, NUM_ORDER_ITEMS + 1)

    # Order item ID
    .withColumnRenamed("id", "order_item_id")

    # Connect to an existing order
    .withColumn(
        "order_id",
        (F.rand(seed=70) * NUM_ORDERS + 1).cast("long")
    )

    # Connect to an existing product
    .withColumn(
        "product_id",
        (F.rand(seed=71) * NUM_PRODUCTS + 1).cast("long")
    )

    # Quantity between 1 and 5
    .withColumn(
        "quantity",
        (F.rand(seed=72) * 5 + 1).cast("int")
    )

    # Unit price
    .withColumn(
        "unit_price",
        F.round(
            F.rand(seed=73) * 4950 + 50,
            2
        )
    )

    # Discount between 0% and 20%
    .withColumn(
        "discount",
        F.round(
            F.rand(seed=74) * 0.20,
            2
        )
    )

    # Last updated timestamp
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(order_items.limit(20))

In [0]:
duplicate_customers = customers.limit(100)

customers_with_duplicates = customers.unionByName(
    duplicate_customers
)

print("Original customer count:", customers.count())
print("After adding duplicates:", customers_with_duplicates.count())

display(customers_with_duplicates.limit(10))

In [0]:
bad_customers = (
    spark.range(1, 21)
    .select(
        F.lit(None).cast("long").alias("customer_id"),
        F.concat(F.lit("BadCustomer_"), F.col("id")).alias("customer_name"),
        F.lit("invalid-email").alias("email"),
        F.lit("Pune").alias("city"),
        F.lit("Maharashtra").alias("state"),
        F.lit("India").alias("country"),
        F.current_date().alias("signup_date"),
        F.current_timestamp().alias("updated_at")
    )
)

customers_with_issues = customers_with_duplicates.unionByName(
    bad_customers
)

print("Customers after introducing DQ issues:",
      customers_with_issues.count())

display(customers_with_issues.filter(
    F.col("customer_id").isNull()
))

In [0]:
bad_orders = (
    spark.range(1, 21)
    .select(
        (F.col("id") + NUM_ORDERS).cast("long").alias("order_id"),
        (F.rand(seed=80) * NUM_CUSTOMERS + 1).cast("long").alias("customer_id"),
        F.current_date().alias("order_date"),
        F.lit("UNKNOWN").alias("order_status"),
        F.round(F.rand(seed=81) * 9950 + 50, 2).alias("total_amount"),
        F.current_timestamp().alias("updated_at")
    )
)

orders_with_issues = orders.unionByName(bad_orders)

print("Original order count:", orders.count())
print("Orders after DQ issues:", orders_with_issues.count())

display(
    orders_with_issues
    .filter(F.col("order_status") == "UNKNOWN")
)

In [0]:
bad_order_items = (
    spark.range(1, 21)
    .select(
        (F.col("id") + NUM_ORDER_ITEMS).cast("long").alias("order_item_id"),

        # Existing orders
        (F.rand(seed=90) * NUM_ORDERS + 1).cast("long").alias("order_id"),

        # Deliberately invalid product ID
        F.lit(999999).cast("long").alias("product_id"),

        # Deliberately invalid quantity
        F.lit(0).cast("int").alias("quantity"),

        # Unit price
        F.round(
            F.rand(seed=91) * 4950 + 50,
            2
        ).alias("unit_price"),

        # Discount
        F.lit(0.10).alias("discount"),

        F.current_timestamp().alias("updated_at")
    )
)

order_items_with_issues = order_items.unionByName(
    bad_order_items
)

print("Original order item count:", order_items.count())
print("Order items after DQ issues:",
      order_items_with_issues.count())

display(
    order_items_with_issues
    .filter(F.col("product_id") == 999999)
)